# EverFlavor AI - Ingredient Knowledge: Names, Substitutions, Pairings and Shelf Life

**Team:** EverGlow
**Course:** ITAI 2277
**Project:** EverFlavor AI - Multi-Agent Recipe and Nutrition System

> **Status: planning sketch.** The sections, sources, output tables and functions below are the plan.
> The functions are stubs (`NotImplementedError`) and nothing here has been run or tested yet.
> "Run all" is safe: it only runs the setup and prints the plan's status.

---

## Purpose

Notebook 01 knows each recipe's ingredients by one English name. The agents need to know more about
each ingredient:

- **Names in other languages and regions** ("coriander" / "cilantro" / "dhania"; "aubergine" / "eggplant";
  "bell pepper" / "capsicum"). This is what makes adding a new region cheap: a recipe or a store label in
  another language maps to the same ingredient.
- **Substitutions:** what to use when an ingredient is missing or not allowed, and what changes (flavor,
  texture, allergens).
- **Pairings:** which ingredients are often used together, and in which cuisines.
- **Shelf life:** how long an ingredient keeps in the pantry, fridge and freezer.

**Safety rule:** a substitution is only suggested after the substitute is checked against the person's
restrictions with the same flag rules as notebook 01 (`everflavor.flags`). A substitution never removes a
caution on its own.

## Inputs and outputs

| | File | Made by |
|---|---|---|
| Input | `data/interim/recipes_all.parquet` (ingredient lists, cuisines, countries) | Notebook 01 |
| Input | Food.com `RAW_interactions.csv` (review text), reused from the kagglehub cache | Notebook 01 (section 2.4.2) |
| Input | Wikidata (SPARQL query service, CC0, no key) | Section 2 |
| Input | USDA FoodKeeper data (public domain) | Section 5 |
| Output | `data/processed/ingredient_names.parquet` | Section 2 |
| Output | `data/processed/ingredient_substitutions.parquet` | Section 3 |
| Output | `data/processed/ingredient_pairings.parquet` | Section 4 |
| Output | `data/processed/ingredient_shelf_life.parquet` | Section 5 |

## Contents

1. Setup
2. Ingredient Names Across Languages (Wikidata)
3. Substitutions (Food.com reviews)
4. Pairings (from our own recipes)
5. Shelf Life (USDA FoodKeeper)
6. Team Decisions and Next Steps

---

## 1. Setup

The same setup as notebooks 01 and 02: it works unchanged in Google Colab, VS Code and Antigravity.

In [ ]:
# %pip (not !pip) installs into the Python that runs this notebook,
# in Colab and in local kernels (VS Code / Antigravity / JupyterLab).
%pip install -q kagglehub requests pandas pyarrow

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd

# -------------------------------------------------------
# 1. Where is this notebook running? Move to the project folder
#    (the one that holds notebooks/ and data/), as in notebook 01.
# -------------------------------------------------------
try:
    import google.colab  # noqa: F401  # pyright: ignore[reportMissingImports]  (only exists in Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def find_project_root(start):
    # Return the nearest folder at or above `start` with notebooks/ and data/, or None
    start = Path(start).resolve()
    for folder in [start, *start.parents]:
        if (folder / "notebooks").is_dir() and (folder / "data").is_dir():
            return folder
    return None

PROJECT_ROOT = find_project_root(Path.cwd())
if PROJECT_ROOT is None and IN_COLAB and Path("/content/The-EverFlavor-AI").is_dir():
    PROJECT_ROOT = Path("/content/The-EverFlavor-AI")
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd()
os.chdir(PROJECT_ROOT)

# -------------------------------------------------------
# 2. The project's code (src/everflavor). If the notebook was opened on its
#    own, the missing files are downloaded from GitHub (temporary name
#    first, __init__.py last), exactly as in notebook 01.
# -------------------------------------------------------
SRC_DIR = PROJECT_ROOT / "src"
EVERFLAVOR_MODULES = ["__init__", "checks", "environment", "charts", "sources", "parsing", "cuisine",
                      "ingredients", "flags", "diets", "nutrition", "features", "pipeline", "review",
                      "recommend", "reporting", "cooking"]
EVERFLAVOR_URL = "https://raw.githubusercontent.com/Chloe-Tu2/The-EverFlavor-AI/main/src/everflavor/"
EVERFLAVOR_DIR = SRC_DIR / "everflavor"
missing_modules = [m for m in EVERFLAVOR_MODULES if not (EVERFLAVOR_DIR / f"{m}.py").exists()]
if missing_modules:
    import urllib.request
    EVERFLAVOR_DIR.mkdir(parents=True, exist_ok=True)
    for module in sorted(missing_modules, key=lambda m: m == "__init__"):
        partial = EVERFLAVOR_DIR / f"{module}.py.part"
        urllib.request.urlretrieve(f"{EVERFLAVOR_URL}{module}.py", partial)
        partial.replace(EVERFLAVOR_DIR / f"{module}.py")
    print(f"Downloaded {len(missing_modules)} project code file(s) (src/everflavor) from GitHub.")
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from everflavor.environment import short_path

REFRESH_DOWNLOADS = False   # True downloads every source again instead of using the saved copy
SPLIT_FILES = {s: Path(f"data/processed/recipes_{s}.parquet") for s in ["train", "val", "test"]}
ALL_RECIPES_FILE = Path("data/interim/recipes_all.parquet")
WIKIDATA_CACHE = Path("data/raw/wikidata_cache")
FOODKEEPER_DIR = Path("data/raw/foodkeeper")
for folder in [WIKIDATA_CACHE, FOODKEEPER_DIR, Path("data/processed")]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Runtime        : {'Google Colab' if IN_COLAB else 'Local Jupyter (VS Code / Antigravity / JupyterLab)'}")
print(f"Python         : {sys.version.split()[0]}")
print(f"Project folder : {short_path(PROJECT_ROOT)}")
missing_inputs = [str(p) for p in [*SPLIT_FILES.values(), ALL_RECIPES_FILE] if not p.exists()]
if missing_inputs:
    print("\nMISSING notebook 01 outputs:", ", ".join(missing_inputs))
    print("Run notebook 01 first (all of Week 5), then run this notebook again.")
else:
    print("Notebook 01 outputs found.")

---

## 2. Ingredient Names Across Languages (Wikidata)

**Source:** Wikidata, through its SPARQL query service. Wikidata is **CC0** (no restrictions). Each food
item has labels and aliases in many languages ("Coriandrum sativum": cilantro, coriander, dhania, 香菜,
كزبرة ...).

**Steps:**

1. Take the ingredient names used in at least N recipes (the same threshold idea as section 5.13).
2. Look up each name's Wikidata item by English label or alias; ambiguous matches ("pepper": the spice
   or the vegetable?) go to a **hand-check list**, like the flag review.
3. Download labels and aliases in the pilot languages (team decision, section 6).

**Planned table** (`ingredient_names.parquet`): `ingredient` (our name), `wikidata_id`, `language`,
`name`, `kind` (`label` / `alias`), `match` (`exact` / `hand-checked` / `ambiguous`).

**Limits:** the query service asks for a descriptive User-Agent and has a 60-second query limit, so names
are queried in batches and cached.

In [ ]:
# Planned module: src/everflavor/knowledge.py
WIKIDATA_SPARQL = "https://query.wikidata.org/sparql"
PILOT_LANGUAGES = ["en", "es", "fr", "it", "hi", "zh", "ja", "ar", "am", "ko", "vi", "fa"]  # team decision


def wikidata_matches(names: list[str], cache_dir: Path, refresh: bool = False) -> pd.DataFrame:
    """Find candidate Wikidata items for each English ingredient name (label or alias match), cached.

    Returns ingredient, wikidata_id, description, n_candidates. More than one candidate -> hand check.
    """
    raise NotImplementedError("sketch: Wikidata lookup")


def names_in_languages(wikidata_ids: list[str], languages: list[str], cache_dir: Path) -> pd.DataFrame:
    """Labels and aliases of the matched items in `languages`."""
    raise NotImplementedError("sketch: Wikidata labels")

---

## 3. Substitutions (Food.com Reviews)

**Source:** Food.com reviews (`RAW_interactions.csv`, about 1.1 million reviews, already downloaded by
notebook 01). Reviewers often write what they changed: "I used Greek yogurt instead of sour cream",
"substituted honey for the sugar".

**Steps:**

1. Find substitution sentences with patterns ("used X instead of Y", "substituted X for Y", "swapped Y
   for X", "replaced Y with X").
2. Normalize X and Y with the same ingredient cleaner as notebook 01, and keep a pair only when Y is in
   that recipe's ingredient list (this filters out many wrong matches).
3. Count each pair and the average rating of reviews that used it.
4. Add what the swap changes: the restriction flags of X and Y (for example sour cream -> coconut cream
   removes dairy) and their nutrition from the USDA table of notebook 01.

**Planned table** (`ingredient_substitutions.parquet`): `original`, `substitute`, `n_reviews`,
`mean_rating`, `flags_removed`, `flags_added`, `example_review_id`.

**Hand check:** a blind sample of pairs, as with the flags, before any agent uses them.

In [ ]:
SUBSTITUTION_PATTERNS = [
    r"used (?P<new>[a-z \-]+?) instead of (?:the )?(?P<old>[a-z \-]+)",
    r"substituted (?P<new>[a-z \-]+?) for (?:the )?(?P<old>[a-z \-]+)",
    r"replaced (?:the )?(?P<old>[a-z \-]+?) with (?P<new>[a-z \-]+)",
    r"swapped (?:the )?(?P<old>[a-z \-]+?) for (?P<new>[a-z \-]+)",
]


def substitutions_from_reviews(reviews: pd.DataFrame, recipes: pd.DataFrame) -> pd.DataFrame:
    """Find "X instead of Y" pairs in review text, keeping only pairs where Y is in the recipe."""
    raise NotImplementedError("sketch: substitution mining")


def substitution_effects(pairs: pd.DataFrame) -> pd.DataFrame:
    """Add flags_removed and flags_added for each pair, using everflavor.flags on both names."""
    raise NotImplementedError("sketch: substitution effects")

---

## 4. Pairings (From Our Own Recipes)

No new download: the 291,771 recipes from notebook 01 already show which ingredients are used together.
For each pair of ingredients the score is **PMI** (pointwise mutual information): how much more often
they appear together than by chance. Computed per cuisine family and per country, so "ginger + soy sauce"
scores high in Asian recipes and "basil + tomato" in Italian ones.

**Planned table** (`ingredient_pairings.parquet`): `ingredient_a`, `ingredient_b`, `scope` (`all`, a
family or a country), `n_recipes`, `pmi`.

Only pairs seen in at least a minimum number of recipes are kept, so rare pairs do not get huge scores.

In [ ]:
def ingredient_pairings(recipes: pd.DataFrame, scope_column: str | None = None,
                        min_recipes: int = 30) -> pd.DataFrame:
    """PMI of ingredient pairs, overall or within each value of `scope_column`."""
    raise NotImplementedError("sketch: pairing scores")

---

## 5. Shelf Life (USDA FoodKeeper)

**Source:** USDA FoodKeeper (Food Safety and Inspection Service), public domain. For about 650 foods it
gives storage times in the pantry, refrigerator and freezer, before and after opening, with tips.

**Steps:** download the FoodKeeper data file (the exact file is to be confirmed), turn its min / max
times and units into days, and match FoodKeeper food names to our ingredient names (hand-check the
matches, like the USDA nutrition matches in notebook 01).

**Planned table** (`ingredient_shelf_life.parquet`): `ingredient`, `foodkeeper_id`, `storage`
(`pantry` / `fridge` / `freezer`), `opened`, `min_days`, `max_days`, `tip`.

**Wording rule:** times are **guidance from USDA**, not a guarantee that food is safe ("USDA suggests
using it within 3-5 days").

In [ ]:
def foodkeeper_table(folder: Path, refresh: bool = False) -> pd.DataFrame:
    """Download and tidy the USDA FoodKeeper data: one row per food and storage place, times in days."""
    raise NotImplementedError("sketch: FoodKeeper download")

---

## 6. Team Decisions and Next Steps

1. **Pilot languages** (`PILOT_LANGUAGES`): should follow the pilot countries chosen in notebook 03.
2. **Who hand-checks** the ambiguous Wikidata matches and the sample of substitution pairs.
3. **Minimum recipe counts** for names (section 2) and pairings (section 4).

**Build order:** pairings first (no download), then substitutions (data already on disk), then shelf
life, then names.

---

## Plan status

What is built, what is a stub, and what needs a team decision first.

In [ ]:
PLAN = pd.DataFrame([
    ("Plan, sources, output tables", "written"),
    ("2 Names across languages (Wikidata)", "stub - needs pilot languages"),
    ("3 Substitutions from reviews", "stub - data already downloaded"),
    ("4 Pairings (PMI)", "stub - build first, no download"),
    ("5 Shelf life (FoodKeeper)", "stub - confirm data file"),
], columns=["step", "status"])
print(PLAN.to_string(index=False))